# Ultralytics Hand Keypoints: 기존 주석 보존 + MediaPipe handedness 추가
Ultralytics가 배포하는 **Hand Keypoints 데이터셋**을 이 노트북에서 직접 다운로드합니다. 이 데이터셋의 기존 키포인트와 새 MediaPipe 추론 좌표가 충분히 일치하는 이미지만 채택합니다.
원본은 MediaPipe로 자동 주석된 hand 1클래스, 손 관절 21개 데이터셋이며 Google의 MediaPipe 학습 원본 데이터셋을 뜻하지 않습니다.
**원본 bbox, 21개 키포인트 x/y, visibility(0/1/2)는 그대로 보존**하고 클래스만 0=left_hand, 1=right_hand로 변경합니다.
MediaPipe 좌표로 원본을 덮어쓰지 않습니다. 원본 이미지 파일도 변형 없이 복사합니다.

가까운 예측끼리 일대일 대응시키며, 원본 손과 검출 손 개수가 다르거나 어느 손이든 기준을 통과하지 못하면 이미지 전체를 제외합니다.
v=0 관절은 비교에서 제외하고 v=1/2 관절을 비교합니다. 보이지 않는 손가락을 v=0으로 임의 변경하지 않습니다.
일치도는 정답 정확도의 보증이 아닙니다. 두 결과가 같은 오류를 가질 수 있으므로 검수하세요.
보존된 visibility의 의미는 원본 데이터의 라벨 정책을 따르며, YOLO 기본 학습은 v=1과 v=2를 모두 유효 관절로 취급합니다.

출처: [Ultralytics Hand Keypoints 데이터셋](https://docs.ultralytics.com/datasets/pose/hand-keypoints/),
[공식 모델 안내](https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker).
원본 데이터 이용 조건(CC BY-NC-SA 4.0)을 유지합니다.

## 1. 설치
Colab의 새 Python 런타임에서 실행하세요. 이미 MediaPipe/NumPy를 import한 상태에서 업그레이드했다면 런타임을 재시작한 뒤 실행하세요.
설치 버전과 실제 모델 SHA-256을 결과에 기록합니다.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "mediapipe", "Pillow", "PyYAML", "tqdm"])

from pathlib import Path, PurePosixPath
from urllib.request import urlopen
from urllib.parse import urljoin, urlparse
from html.parser import HTMLParser
from datetime import datetime, timezone
from collections import Counter
from itertools import permutations
import hashlib
import json
import math
import shutil
import zipfile

import mediapipe as mp
from mediapipe.tasks.python import vision
import numpy as np
import yaml
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from google.colab import files, drive

print("MediaPipe:", mp.__version__)

## 2. 설정
입력 ZIP을 준비할 필요가 없습니다. 3번 셀이 Ultralytics 공식 배포 URL에서 Hand Keypoints 데이터셋을 직접 다운로드합니다. Drive는 결과 저장용으로만 마운트합니다.
비교 거리는 픽셀 유클리드 거리를 **원본 손 bbox의 픽셀 대각선 길이**로 나눕니다. 영상 종횡비를 반영합니다.

- MAX_MEAN_DISTANCE=0.05: v>0 관절의 평균 차이 ≤ bbox 대각선의 5%.
- MAX_POINT_DISTANCE=0.15: 개별 관절 최대 차이 ≤ 15%.
- MIN_COMPARE_POINTS=5: 비교 가능한 관절이 최소 5개.
- MATCH_AMBIGUITY_MARGIN=0.01: 두 손의 대안 매칭 평균 비용 차이가 1% 미만이면 모호한 매칭으로 제외.
- MIN_HANDEDNESS_SCORE=0.8: 좌우 분류 점수 기준. 위치 품질 점수가 아닙니다.
위 수치는 시작값입니다. 미리보기의 차이와 채택률을 보며 조정하세요.
SWAP_HANDEDNESS는 알려진 손의 좌우로 검증한 뒤 필요할 때만 변경합니다.
이미지 좌표를 보존하기 위해 EXIF 자동 회전·좌우 반전·리사이즈를 하지 않습니다.

In [ ]:
SAVE_TO_DRIVE = True
MAX_IMAGES_PER_SPLIT = 0
MAX_HANDS = 2
MIN_DETECTION_CONFIDENCE = 0.5
MIN_PRESENCE_CONFIDENCE = 0.5
MIN_HANDEDNESS_SCORE = 0.8
SWAP_HANDEDNESS = False
MAX_MEAN_DISTANCE = 0.05
MAX_POINT_DISTANCE = 0.15
MIN_COMPARE_POINTS = 5
MATCH_AMBIGUITY_MARGIN = 0.01
SEED = 42

assert MAX_IMAGES_PER_SPLIT >= 0 and 1 <= MAX_HANDS <= 4
assert all(0 <= v <= 1 for v in (
    MIN_DETECTION_CONFIDENCE, MIN_PRESENCE_CONFIDENCE, MIN_HANDEDNESS_SCORE))
assert 1 <= MIN_COMPARE_POINTS <= 21
assert 0 <= MAX_MEAN_DISTANCE <= MAX_POINT_DISTANCE and MATCH_AMBIGUITY_MARGIN >= 0
if SAVE_TO_DRIVE:
    drive.mount("/content/drive")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
WORK = Path("/content/hand_relabel") / RUN_ID
WORK.mkdir(parents=True, exist_ok=False)
OUT = WORK / "hand_mediapipe_relabel"
NAMES = {0: "left_hand", 1: "right_hand"}
KEYPOINT_NAMES = ["wrist", "thumb_cmc", "thumb_mcp", "thumb_ip", "thumb_tip",
    "index_mcp", "index_pip", "index_dip", "index_tip",
    "middle_mcp", "middle_pip", "middle_dip", "middle_tip",
    "ring_mcp", "ring_pip", "ring_dip", "ring_tip",
    "pinky_mcp", "pinky_pip", "pinky_dip", "pinky_tip"]
EDGES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),
    (15,16),(13,17),(0,17),(17,18),(18,19),(19,20)]
DATASET_NAME = "Ultralytics Hand Keypoints"
DATASET_DOCS = "https://docs.ultralytics.com/datasets/pose/hand-keypoints/"
DATA_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/hand-keypoints.zip"
GUIDE_URL = "https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker"

## 3. Ultralytics Hand Keypoints 자동 다운로드 및 모델 준비
공식 안내의 최신 모델 링크를 확인하고 실제 URL·해시를 기록합니다.
원본 ZIP에서 images/train,val과 대응하는 labels/train,val을 읽습니다. 기존 라벨은 비교 및 출력 보존에 사용합니다.
라벨 누락/파싱 오류는 이미지별 제외 사유로 기록합니다.

In [ ]:
def download(url, target):
    partial = target.with_suffix(target.suffix + ".part")
    try:
        with urlopen(url, timeout=120) as response, partial.open("wb") as output:
            headers = dict(response.headers)
            final_url = response.geturl()
            shutil.copyfileobj(response, output)
        if partial.stat().st_size == 0:
            raise ValueError("빈 다운로드")
        partial.replace(target)
        return {"requested_url": url, "resolved_url": final_url,
                "etag": headers.get("ETag"), "last_modified": headers.get("Last-Modified")}
    finally:
        partial.unlink(missing_ok=True)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

class ModelLinks(HTMLParser):
    def __init__(self):
        super().__init__()
        self.href = None
        self.text_parts = []
        self.links = []
    def handle_starttag(self, tag, attrs):
        if tag == "a":
            self.href = dict(attrs).get("href")
            self.text_parts = []
    def handle_data(self, data):
        if self.href:
            self.text_parts.append(data)
    def handle_endtag(self, tag):
        if tag == "a" and self.href:
            self.links.append((self.href, "".join(self.text_parts).strip()))
            self.href = None

with urlopen(GUIDE_URL, timeout=60) as response:
    guide_html = response.read().decode("utf-8")
(WORK / "model_guide.html").write_text(guide_html, encoding="utf-8")
parser = ModelLinks()
parser.feed(guide_html)
candidates = [(urljoin(GUIDE_URL, href), label) for href, label in parser.links
    if urlparse(urljoin(GUIDE_URL, href)).hostname == "storage.googleapis.com"
    and "/hand_landmarker/" in href and href.endswith("hand_landmarker.task")]
if not candidates:
    raise RuntimeError("공식 모델 링크를 찾지 못했습니다. 공식 안내 페이지를 확인하세요.")
MODEL_URL = next((url for url, label in candidates if "latest" in label.lower()), candidates[0][0])
MODEL_PATH = WORK / "hand_landmarker.task"
model_download = download(MODEL_URL, MODEL_PATH)
model_hash = file_sha256(MODEL_PATH)
print("모델:", MODEL_URL, "\nSHA-256:", model_hash)

source_zip = WORK / "hand-keypoints.zip"
print("데이터셋 다운로드:", DATASET_NAME, DATA_URL)
dataset_download = download(DATA_URL, source_zip)
if not source_zip.is_file() or not zipfile.is_zipfile(source_zip):
    raise ValueError("유효한 원본 ZIP 파일이 필요합니다.")

source_dir = WORK / "source_images"
source_dir.mkdir()
sources = {"train": [], "val": []}
source_labels = {}
suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
with zipfile.ZipFile(source_zip) as archive:
    archive_names = archive.namelist()
    if len(archive_names) != len(set(archive_names)):
        raise ValueError("중복 ZIP 항목이 있습니다.")
    for info in tqdm(archive.infolist(), desc="이미지 및 원본 라벨 추출"):
        member = PurePosixPath(info.filename)
        if info.is_dir() or member.suffix.lower() not in suffixes:
            continue
        parts = member.parts
        positions = [i for i, part in enumerate(parts[:-2])
                     if part == "images" and parts[i+1] in sources]
        if len(positions) != 1:
            continue
        split = parts[positions[0]+1]
        if member.is_absolute() or ".." in parts or "\\" in info.filename:
            raise ValueError(f"안전하지 않은 ZIP 경로: {info.filename}")
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("이미지 심볼릭 링크는 지원하지 않습니다.")
        target = source_dir.joinpath(*parts).resolve()
        if not target.is_relative_to(source_dir.resolve()) or target.exists():
            raise ValueError(f"중복 또는 외부 경로: {info.filename}")
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(info) as src, target.open("wb") as dst:
            shutil.copyfileobj(src, dst)
        label_parts = list(parts)
        label_parts[positions[0]] = "labels"
        label_member = PurePosixPath(*label_parts).with_suffix(".txt").as_posix()
        label_target = source_dir.joinpath(*PurePosixPath(label_member).parts).resolve()
        if not label_target.is_relative_to(source_dir.resolve()):
            raise ValueError("외부 라벨 경로")
        if label_member in archive_names:
            label_info = archive.getinfo(label_member)
            if (label_info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("라벨 심볼릭 링크는 지원하지 않습니다.")
            label_target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(label_info) as src, label_target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
        source_labels[target] = label_target
        sources[split].append(target)
for split, paths in sources.items():
    paths.sort()
    if not paths:
        raise ValueError(f"images/{split} 이미지가 없습니다.")
    if MAX_IMAGES_PER_SPLIT:
        rng = np.random.default_rng(SEED)
        chosen = rng.choice(len(paths), min(MAX_IMAGES_PER_SPLIT, len(paths)), replace=False)
        sources[split] = [paths[i] for i in sorted(chosen)]
print({split: len(paths) for split, paths in sources.items()})

## 4. 원본 라벨 보존 및 일대일 매칭
각 원본 손의 v>0 관절과 동일 번호의 예측 관절을 비교합니다.
손 순서가 뒤바뀌어도 전체 일대일 할당을 비교하여 통과 가능한 매칭 중 평균 오차가 가장 작은 것을 선택합니다.
모호한 매칭, 손 개수 차이, 평균/최대 오차 초과, 좌우 점수 미달은 이미지 전체 제외입니다.
출력 라벨은 원본 행의 첫 토큰(클래스)만 바꾸며 나머지 토큰은 그대로 저장합니다.

In [ ]:
def read_rgb(path):
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB")).copy()

def parse_original(text):
    hands = []
    for row, line in enumerate(text.splitlines(), 1):
        tokens = line.split()
        if not tokens:
            continue
        values = [float(t) for t in tokens]
        if len(values) != 68 or not all(math.isfinite(v) for v in values):
            raise ValueError(f"{row}: expected 68 finite values")
        if not values[0].is_integer() or values[0] < 0:
            raise ValueError(f"{row}: invalid class")
        cx, cy, bw, bh = values[1:5]
        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
            raise ValueError(f"{row}: invalid bbox")
        points = [values[i:i+3] for i in range(5, 68, 3)]
        if any(not (0 <= x <= 1 and 0 <= y <= 1 and v in (0,1,2)) for x,y,v in points):
            raise ValueError(f"{row}: invalid keypoint or visibility")
        hands.append({"tokens": tokens, "bbox": values[1:5], "points": points, "source_row": row})
    return hands

def pair_distance(original, predicted, width, height, min_points):
    indices = [i for i,p in enumerate(original["points"]) if p[2] > 0]
    if len(indices) < min_points:
        return None
    diagonal = math.hypot(original["bbox"][2]*width, original["bbox"][3]*height)
    distances = [math.hypot((original["points"][i][0]-predicted["xyz"][i][0])*width,
                           (original["points"][i][1]-predicted["xyz"][i][1])*height) / diagonal
                 for i in indices]
    return {"mean": sum(distances)/len(distances), "max": max(distances),
            "compared_points": len(indices), "per_point": dict(zip(indices, distances))}

def match_hands(originals, predictions, width, height, mean_limit, point_limit,
                min_points, ambiguity_margin, min_score, swap=False):
    n = len(originals)
    if not n:
        return [], "empty_original_label", {}
    if len(predictions) != n:
        return [], "hand_count_mismatch", {"original_count": n, "predicted_count": len(predictions)}
    if n > 4:
        return [], "too_many_hands", {}
    costs = [[pair_distance(o,p,width,height,min_points) for p in predictions] for o in originals]
    if any(all(metric is None for metric in row) for row in costs):
        return [], "insufficient_original_points", {}
    ranked = []
    for order in permutations(range(n)):
        metrics = [costs[i][j] for i,j in enumerate(order)]
        if any(m is None for m in metrics):
            continue
        ranked.append((sum(m["mean"] for m in metrics)/n, order, metrics))
    ranked.sort(key=lambda x:x[0])
    feasible = [r for r in ranked if all(
        m["mean"] <= mean_limit and m["max"] <= point_limit for m in r[2])]
    if not feasible:
        return [], "keypoint_mismatch", {"best_mean_distance": ranked[0][0],
            "best_pair_metrics": ranked[0][2]}
    best = feasible[0]
    alternatives = [r for r in ranked if r[1] != best[1]]
    if alternatives and abs(alternatives[0][0]-best[0]) < ambiguity_margin:
        return [], "ambiguous_hand_match", {"best_mean_distance": best[0],
                                            "alternative_mean_distance": alternatives[0][0]}
    output = []
    for i, j in enumerate(best[1]):
        original, pred = originals[i], predictions[j]
        name, score = pred["name"], pred["score"]
        if name not in ("Left", "Right"):
            return [], "unknown_handedness", {}
        if not math.isfinite(score) or not min_score <= score <= 1:
            return [], "low_handedness_score", {"source_row": original["source_row"], "score": score if math.isfinite(score) else None}
        if swap:
            name = {"Left":"Right", "Right":"Left"}[name]
        cls = {"Left":0, "Right":1}[name]
        output.append({"class_id":cls, "handedness":NAMES[cls],
            "raw_handedness":pred["name"], "handedness_score":score,
            "source_row":original["source_row"], "matched_prediction":j,
            "landmarks_xy":[p[:2] for p in original["points"]],
            "visibility":[int(p[2]) for p in original["points"]],
            "predicted_landmarks_xyz":pred["xyz"], "landmark_confidence":None,
            "bbox_xywh":original["bbox"], "match_metrics":best[2][i],
            "yolo_line":str(cls)+" "+" ".join(original["tokens"][1:])})
    return output, None, {"mean_distance":best[0]}

def make_detector():
    return vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=vision.RunningMode.IMAGE, num_hands=MAX_HANDS,
        min_hand_detection_confidence=MIN_DETECTION_CONFIDENCE,
        min_hand_presence_confidence=MIN_PRESENCE_CONFIDENCE))

def infer_image(detector, rgb, originals):
    if not originals:
        return [], "empty_original_label", {}
    if len(originals) > MAX_HANDS:
        return [], "too_many_original_hands", {}
    result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)))
    if not result.hand_landmarks:
        return [], "no_hand", {}
    if len(result.hand_landmarks) != len(result.handedness):
        raise RuntimeError("Landmark and handedness counts differ")
    predictions = []
    for landmarks, categories in zip(result.hand_landmarks, result.handedness):
        if not categories:
            return [], "missing_handedness", {}
        xyz = [[float(p.x),float(p.y),float(p.z)] for p in landmarks]
        if len(xyz) != 21 or not all(math.isfinite(v) for p in xyz for v in p):
            return [], "invalid_prediction", {}
        category = max(categories, key=lambda c:c.score)
        predictions.append({"xyz":xyz, "name":category.category_name, "score":float(category.score)})
    h,w = rgb.shape[:2]
    return match_hands(originals,predictions,w,h,MAX_MEAN_DISTANCE,MAX_POINT_DISTANCE,
                       MIN_COMPARE_POINTS,MATCH_AMBIGUITY_MARGIN,MIN_HANDEDNESS_SCORE,SWAP_HANDEDNESS)

def draw_hands(ax, rgb, hands):
    ax.imshow(rgb)
    h,w = rgb.shape[:2]
    for hand in hands:
        points = np.asarray(hand["landmarks_xy"])*[w,h]
        predicted = np.asarray(hand["predicted_landmarks_xyz"])[:,:2]*[w,h]
        mask = np.asarray(hand["visibility"]) > 0
        for a,b in EDGES:
            if mask[a] and mask[b]:
                ax.plot(points[[a,b],0],points[[a,b],1],color="lime",linewidth=1)
        ax.scatter(points[mask,0],points[mask,1],c="lime",s=12,label="original")
        ax.scatter(predicted[mask,0],predicted[mask,1],c="magenta",s=10,marker="x",label="inferred")
        for i in np.flatnonzero(mask):
            ax.text(*points[i],str(i),color="yellow",fontsize=7)
        cx,cy,bw,bh = hand["bbox_xywh"]
        ax.text((cx-bw/2)*w,max(0,(cy-bh/2)*h),
                f'{hand["handedness"]} {hand["handedness_score"]:.2f} / mean {hand["match_metrics"]["mean"]:.3f}',
                color="white",bbox=dict(facecolor="black",alpha=.6))
    ax.set_xlim(0,w)
    ax.set_ylim(h,0)
    ax.axis("off")

## 5. 소량 미리보기
초록: 유지할 원본 관절. 자홍 ×: 새 추론 좌표. 클래스는 새 handedness입니다.
제외 사유는 제목에 표시합니다. 채택 기준은 시작값이며 실제 검수에 맞게 조정하세요.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
rng = np.random.default_rng(SEED)
pool = sources["train"]
indices = rng.choice(len(pool), size=min(6, len(pool)), replace=False)
for ax in axes.flat:
    ax.axis("off")
with make_detector() as detector:
    for ax, index in zip(axes.flat, indices):
        path = pool[index]
        rgb = read_rgb(path)
        try:
            originals = parse_original(source_labels[path].read_text(encoding="utf-8-sig"))
            hands, reason, diagnostics = infer_image(detector, rgb, originals)
        except (OSError, ValueError) as exc:
            hands, reason = [], "original_label_error"
            print(path.name, exc)
        draw_hands(ax, rgb, hands)
        ax.set_title(path.name + (" / 제외: " + reason if reason else ""))
plt.tight_layout()
plt.show()

## 6. 전체 필터링 및 handedness 추가
원본 이미지 바이트를 그대로 복사하고, 라벨은 클래스 토큰만 교체합니다.
다른 원본 bbox/좌표/visibility 값을 수정하지 않습니다. v=0 좌표도 그대로 보존합니다.
모든 손이 대응되어야 이미지를 채택하며, 제외 사유 및 평균/최대 거리와 매칭 인덱스를 manifest.jsonl에 기록합니다.
동일 RGB 픽셀 중복은 한 번만 채택합니다.

In [ ]:
# 재실행해도 이전 결과에 라벨을 덧붙이지 않습니다.
OUT = WORK / ("hand_mediapipe_relabel_" + datetime.now(timezone.utc).strftime("%H%M%S_%f"))
for split in sources:
    (OUT / "images" / split).mkdir(parents=True, exist_ok=False)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=False)
stats = {split: Counter() for split in sources}
class_counts = {split: Counter() for split in sources}
seen_pixels = {}
examples = []
complete = False
validated = False
with (OUT / "manifest.jsonl").open("w", encoding="utf-8") as log, make_detector() as detector:
    for split, paths in sources.items():
        for path in tqdm(paths, desc=f"재라벨링 {split}"):
            record = {"source": path.relative_to(source_dir).as_posix(), "split": split}
            stats[split]["processed"] += 1
            try:
                rgb = read_rgb(path)
            except (OSError, ValueError) as exc:
                record.update(status="image_read_error", error=str(exc))
            else:
                digest = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()
                record["pixel_sha256"] = digest
                if digest in seen_pixels:
                    record.update(status="duplicate", duplicate_of=seen_pixels[digest])
                else:
                    try:
                        originals = parse_original(source_labels[path].read_text(encoding="utf-8-sig"))
                        hands, reason, diagnostics = infer_image(detector, rgb, originals)
                    except (OSError, ValueError) as exc:
                        hands, reason, diagnostics = [], "original_label_error", {"error": str(exc)}
                    record["comparison"] = diagnostics
                    if reason:
                        record["status"] = reason
                    else:
                        image_rel = Path("images") / split / (digest + path.suffix.lower())
                        label_rel = Path("labels") / split / (digest + ".txt")
                        shutil.copy2(path, OUT / image_rel)
                        (OUT / label_rel).write_text(
                            "\n".join(hand["yolo_line"] for hand in hands) + "\n", encoding="utf-8")
                        seen_pixels[digest] = image_rel.as_posix()
                        record.update(status="accepted", image=image_rel.as_posix(),
                                      label=label_rel.as_posix(), hands=hands,
                                      source_label=source_labels[path].relative_to(source_dir).as_posix())
                        class_counts[split].update(hand["class_id"] for hand in hands)
                        if len(examples) < 6:
                            examples.append(record)
            stats[split][record["status"]] += 1
            log.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
        log.flush()
complete = True
summary = {"counts": {s: dict(c) for s,c in stats.items()},
           "class_counts": {s: dict(c) for s,c in class_counts.items()}}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("출력 폴더:", OUT)

## 7. 산출물 검사 및 메타데이터
각 train/val에 양손 클래스가 모두 있어야 기존 2클래스 학습 노트북에서 사용할 수 있습니다.
전체 이미지가 적거나 한 클래스가 없다면 임계값과 데이터, 좌우 설정을 확인하세요.
자동 라벨로 만든 val은 MediaPipe 예측과의 일치도를 평가하므로 별도 수동 검증 세트를 권장합니다.

In [ ]:
if not complete:
    raise RuntimeError("재라벨링이 완료되지 않았습니다.")
for split in sources:
    if any(class_counts[split][cls] == 0 for cls in NAMES):
        raise ValueError(f"{split}: 양손 클래스가 필요합니다. 현재 {dict(class_counts[split])}")
    images = sorted((OUT / "images" / split).iterdir())
    labels = sorted((OUT / "labels" / split).glob("*.txt"))
    assert len(images) == len(labels) == stats[split]["accepted"]
    for image in images:
        label = OUT / "labels" / split / (image.stem + ".txt")
        lines = label.read_text().splitlines()
        assert lines
        for line in lines:
            v = [float(x) for x in line.split()]
            assert len(v) == 68 and all(math.isfinite(x) for x in v)
            assert v[0] in NAMES and all(0 <= x <= 1 for x in v[1:5])
            assert v[3] > 0 and v[4] > 0
            assert all(0 <= v[i] <= 1 and 0 <= v[i+1] <= 1 and v[i+2] in (0,1,2)
                       for i in range(5,68,3))
# 클래스 외의 모든 토큰과 이미지 바이트가 원본과 같은지 검사합니다.
with (OUT / "manifest.jsonl").open(encoding="utf-8") as manifest:
    for line in manifest:
        record = json.loads(line)
        if record["status"] != "accepted":
            continue
        original = parse_original((source_dir / record["source_label"]).read_text(encoding="utf-8-sig"))
        exported = parse_original((OUT / record["label"]).read_text(encoding="utf-8-sig"))
        assert len(original) == len(exported)
        assert all(a["tokens"][1:] == b["tokens"][1:] for a,b in zip(original,exported))
        assert file_sha256(source_dir / record["source"]) == file_sha256(OUT / record["image"])
config = {"train": "images/train", "val": "images/val", "names": NAMES,
          "kpt_shape": [21,3], "flip_idx": list(range(21)),
          "kpt_names": {c: KEYPOINT_NAMES for c in NAMES}}
# 절대 경로를 넣지 않고 압축 해제 위치의 data.yaml을 사용합니다.
(OUT / "data.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "mediapipe_version": mp.__version__, "model": model_download, "model_sha256": model_hash,
    "model_guide": GUIDE_URL, "source_zip_sha256": file_sha256(source_zip),
    "dataset_name": DATASET_NAME, "dataset_documentation": DATASET_DOCS,
    "source": DATA_URL, "dataset_download": dataset_download,
    "settings": {"max_hands": MAX_HANDS, "min_detection": MIN_DETECTION_CONFIDENCE,
        "min_presence": MIN_PRESENCE_CONFIDENCE, "min_handedness_score": MIN_HANDEDNESS_SCORE,
        "swap_handedness": SWAP_HANDEDNESS,
        "max_mean_distance": MAX_MEAN_DISTANCE, "max_point_distance": MAX_POINT_DISTANCE,
        "min_compare_points": MIN_COMPARE_POINTS, "match_ambiguity_margin": MATCH_AMBIGUITY_MARGIN,
        "max_images_per_split": MAX_IMAGES_PER_SPLIT, "seed": SEED},
    "visibility_policy": "Original keypoint x/y, bbox and visibility tokens preserved; only class replaced.",
    "landmark_confidence": None, "summary": summary}
(OUT / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "environment.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
shutil.copy2(WORK / "model_guide.html", OUT / "model_guide.html")
(OUT / "README.txt").write_text(
    "Original annotations retained after MediaPipe agreement filtering; only handedness class generated.\n"
    "Classes: 0=left_hand, 1=right_hand. Keypoints: MediaPipe 21-point order.\n"
    "Original visibility values (0/1/2) are preserved unchanged. No per-keypoint confidence is generated.\n"
    "Handedness score is not keypoint confidence. Review anatomical left/right before training.\n"
    "Original image bytes and geometry preserved: no EXIF transpose or horizontal flipping. Disable fliplr/flipud.\n"
    "Use custom_handedness mode in train_yolo26n_hand_pose.ipynb with this ZIP.\n"
    "Source images: Ultralytics Hand Keypoints, CC BY-NC-SA 4.0.\n"
    "https://docs.ultralytics.com/datasets/pose/hand-keypoints/\n"
    "Original source credits: 11k Hands, 2000 Hand Gestures, Gesture Recognition; Rion Dsilva.\n",
    encoding="utf-8")
fig, axes = plt.subplots(2,3,figsize=(16,10))
for ax in axes.flat:
    ax.axis("off")
for ax, record in zip(axes.flat, examples):
    draw_hands(ax, read_rgb(OUT / record["image"]), record["hands"])
plt.tight_layout()
fig.savefig(OUT / "preview.jpg", dpi=120)
plt.show()
validated = True
print("라벨 및 이미지 검사 완료")

## 8. 새 ZIP 저장 / 다운로드
채택된 원본 이미지·bbox·키포인트·visibility와 생성된 handedness 클래스를 저장합니다.
검수 로그에는 제외 사유, 비교 거리와 모델 해시가 포함됩니다.
학습 노트북에서는 DATA_SOURCE="custom_handedness"와 생성 ZIP의 DATA_ZIP 경로를 지정하세요.

In [ ]:
DOWNLOAD_TO_BROWSER = False
if not complete or not validated:
    raise RuntimeError("재라벨링 및 산출물 검사를 먼저 완료하세요.")
archive = Path(shutil.make_archive(str(WORK / OUT.name), "zip", root_dir=OUT))
with zipfile.ZipFile(archive) as z:
    if z.testzip() is not None:
        raise RuntimeError("ZIP 무결성 검사 실패")
print("ZIP:", archive, "bytes:", archive.stat().st_size)
if SAVE_TO_DRIVE:
    destination = Path("/content/drive/MyDrive/hand_pose_datasets")
    destination.mkdir(parents=True, exist_ok=True)
    saved_zip = destination / (RUN_ID + "_" + archive.name)
    shutil.copy2(archive, saved_zip)
    print("학습 노트북 DATA_ZIP =", repr(str(saved_zip)))
if DOWNLOAD_TO_BROWSER:
    files.download(str(archive))